# Timing CPU and GPU inference

Load datasets for testing performance in inference

In [1]:
model_path = "Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras"

x_test_path = 'Data/x_test.npy'
y_test_path = 'Data/y_test.npy'

iterations = 30

In [2]:
# Check if GPU is available
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if not device_name:
  compute = 'CPU'
  timings_path = 'Timings/CPU/'
  print('GPU device not found')
else: 
  compute = 'GPU'
  timings_path = 'Timings/GPU/'
  print('Found GPU at: {}'.format(device_name))

2026-06-09 12:27:12.700088: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-09 12:27:12.730240: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-09 12:27:13.626512: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Found GPU at: /device:GPU:0


I0000 00:00:1781000834.997717  804925 gpu_device.cc:2020] Created device /device:GPU:0 with 6181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [5]:
import os
import sys
import time
import numpy as np
# Load input from .npy file
x_test = np.load(x_test_path)
y_test = np.load(y_test_path)

In [6]:
def test_dut():
    start = time.perf_counter()
    y_dut = model.predict(x_test, verbose=0)
    end = time.perf_counter() - start

    return [y_dut, end]

In [7]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

Run the actual inference

In [8]:
from keras.models import load_model
import hgq.layers
from hgq.utils import trace_minmax

model = load_model(model_path)
# Calibrate datalane in HGQ2-model since it has layers with WRAP
trace_minmax(model, x_test, verbose=True)


I0000 00:00:1781002771.188594  804925 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
                                                                  

dense_0: 455
dense_1: 194
dense_2: 191
dense_3: 161
Total: 1001


1001

In [9]:

timestamp = time.strftime("%Y%m%d_%H%M%S")
nr_samples = x_test.shape[0]
timings = []
for i in range(iterations):    
    # do inference
    result = test_dut()
    y_dut = result[0] 
    timings.append(result[1])

    acc = cal_accuracy(y_dut)
    print(f"\nTime: {result[1]}, Acc: {acc}")

2026-06-09 13:00:08.135714: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f51400059e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-06-09 13:00:08.135727: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-06-09 13:00:08.160764: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-06-09 13:00:08.266673: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92300
I0000 00:00:1781002808.774631  887696 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-06-09 13:00:11.811436: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 5.773803157000657, Acc: 0.7435530120481928


2026-06-09 13:00:16.340145: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 3.2547455269996135, Acc: 0.7435530120481928

Time: 3.2845406849992287, Acc: 0.7435530120481928


2026-06-09 13:00:22.887864: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 3.2124591749998217, Acc: 0.7435530120481928

Time: 3.246435837001627, Acc: 0.7435530120481928

Time: 3.27747278699826, Acc: 0.7435530120481928

Time: 3.2302488690002065, Acc: 0.7435530120481928


2026-06-09 13:00:36.029937: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 3.3134893240021484, Acc: 0.7435530120481928

Time: 3.2241081679967465, Acc: 0.7435530120481928

Time: 3.187690126000234, Acc: 0.7435530120481928

Time: 3.322387771000649, Acc: 0.7435530120481928

Time: 3.2933094859981793, Acc: 0.7435530120481928

Time: 3.2854802229994675, Acc: 0.7435530120481928

Time: 3.3102703869990364, Acc: 0.7435530120481928

Time: 3.2621550210023997, Acc: 0.7435530120481928


2026-06-09 13:01:02.392155: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 3.3076153509973665, Acc: 0.7435530120481928

Time: 3.297734431002027, Acc: 0.7435530120481928

Time: 3.2712910490008653, Acc: 0.7435530120481928

Time: 3.318193651000911, Acc: 0.7435530120481928

Time: 3.297476910000114, Acc: 0.7435530120481928

Time: 3.268602872998599, Acc: 0.7435530120481928

Time: 3.280239714997151, Acc: 0.7435530120481928

Time: 3.3038370789981855, Acc: 0.7435530120481928

Time: 3.2437463339992973, Acc: 0.7435530120481928

Time: 3.2605588730002637, Acc: 0.7435530120481928

Time: 3.255923251999775, Acc: 0.7435530120481928

Time: 3.3380279119992338, Acc: 0.7435530120481928

Time: 3.4259994990025007, Acc: 0.7435530120481928

Time: 3.2215978729982453, Acc: 0.7435530120481928

Time: 3.222339350999391, Acc: 0.7435530120481928


In [11]:
timing_results_path = f"{timings_path}timings_dataset-{nr_samples}_{iterations}iterations_{timestamp}.txt"
np.savetxt(timing_results_path,timings)

In [12]:
# Compute statistics for collected timings (convert to ms)
import json
import numpy as np

arr = np.array(timings)
arr_ms = arr * 1e3

stats = {
    'count': int(arr_ms.size),
    'mean_ms': float(np.mean(arr_ms)),
    'median_ms': float(np.median(arr_ms)),
    'std_ms': float(np.std(arr_ms, ddof=0)),
    'min_ms': float(np.min(arr_ms)),
    'max_ms': float(np.max(arr_ms)),
    'p5_ms': float(np.percentile(arr_ms, 5)),
    'p95_ms': float(np.percentile(arr_ms, 95)),
    'inference-rate (MHz)': float((nr_samples * 1000 / np.median(arr_ms) / 1000000)),
}

# Print summary
print('Timing statistics (ms):')
for k,v in stats.items():
    print(f"{k}: {v}")

# Save JSON summary next to timings file if available
try:
    base = timing_results_path
    out = base.rsplit('.',1)[0] + '_stats.json'
except NameError:
    out = 'timing_stats.json'

with open(out, 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Saved timing summary to: {out}')


Timing statistics (ms):
count: 30
mean_ms: 3359.72602319974
median_ms: 3278.8562509977055
std_ms: 450.57394639557856
min_ms: 3187.690126000234
max_ms: 5773.803157000657
p5_ms: 3216.5715890991123
p95_ms: 3386.41228485103
inference-rate (MHz): 0.2531370503807368
Saved timing summary to: Timings/GPU/timings_dataset-830000_30iterations_20260609_130007_stats.json


[Tensorflow profiler](https://github.com/tensorflow/tensorboard/blob/master/docs/tensorboard_profiling_keras.ipynb) 

In [ ]:
#!pip install -U tensorboard_plugin_profile

In [27]:
import tensorflow as tf
tf.profiler.experimental.start("logs/profile")

model.predict(x_test, batch_size=256)

tf.profiler.experimental.stop()


2026-06-09 11:36:14.375320: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:103] Profiler session initializing.
2026-06-09 11:36:14.375341: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:118] Profiler session started.


3243/3243 ━━━━━━━━━━━━━━━━━━━━ 1s 389us/step


2026-06-09 11:36:15.696351: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:68] Profiler session collecting data.
2026-06-09 11:36:16.363972: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:136] Profiler session tear down.
2026-06-09 11:36:16.366006: I external/local_xla/xla/tsl/profiler/rpc/client/save_profile.cc:150] Collecting XSpace to repository: logs/profile/plugins/profile/2026_06_09_11_36_16/KrissDEV.xplane.pb


In [28]:

# Load the TensorBoard notebook extension.
%load_ext tensorboard
# Launch TensorBoard and navigate to the Profile tab to view performance profile
%tensorboard --logdir=logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 287237), started 0:00:39 ago. (Use '!kill 287237' to kill it.)